In [1]:
from tradepy.data.loader import load_future
from tradepy.config.config import load_config, load_symbols, load_feature_config, load_exclude_config
from tradepy.data.cleaner import add_trading_date_by_gap
from tradepy.data.resampler import resample_ohlcv, daily_ohlcv_cummulative
from tradepy.features.generate import generate_features
from tradepy.features.quality_check import data_quality_report
from tradepy.supervised.load import prepare_all
from tradepy.paths import EXCLUDE_CONFIG, SYSTEM_CONFIG
from tradepy.supervised.target import target_triple_barrier_interday
from tradepy.supervised.pretrain import filter_features
from tradepy.supervised.models import GoldLSTM_L1_Move, GoldLSTM_L2_Dir, GoldGRU_L1_Move, GoldGRU_L2_Dir, BasicLSTM_L1_Move, BasicLSTM_L2_Dir, BasicGRU_L1_Move, BasicGRU_L2_Dir, BasicLSTM_L3_Regression, BasicGRU_L3_Regression
from tradepy.supervised.train import train_walk_forward_2level, grid_search_thresholds, create_lstm_dataset
from tradepy.supervised.posttrain import evaluate_model_classification, tune_threshold_wf, save_trading_model_2level, load_trading_model_3level
from tradepy.paths import MODELS_DIR

In [2]:
symbols = load_symbols()
cfg = load_config()

In [3]:
ASSET = symbols[5]
MINUTES             = cfg['sampling_minutes']        # 240
RETURN_HORIZON_MIN  = cfg['return_horizon_min']      # 2880

In [4]:
(
    model_l1,
    model_l2,
    model_l3,
    sc_features,
    sc_l3,
    features,
    params,
    thresholds,
    df_results
) = load_trading_model_3level(
        BasicLSTM_L1_Move,
        BasicLSTM_L2_Dir,
        BasicLSTM_L3_Regression,
        path=f"{MODELS_DIR}/{ASSET.lower()}/trading_model_lstm_3level",
        device="cuda"
)


🚀 Pipeline 3-Level cargado correctamente desde: E:\Futuro\MLAlgoTrading\TradingBots\Models/gc/trading_model_lstm_3level
📌 Features: 71
📌 Thresholds cargados: True
📁 Resultados cargados: True


In [11]:
(
    model_l1_g,
    model_l2_g,
    model_l3_g,
    sc_features_g,
    sc_l3_g,
    features_g,
    params_g,
    thresholds_g,
    df_results_g
) = load_trading_model_3level(
        BasicGRU_L1_Move,
        BasicGRU_L2_Dir,
        BasicGRU_L3_Regression,
        path=f"{MODELS_DIR}/{ASSET.lower()}/trading_model_gru_3level",
        device="cuda"
)


🚀 Pipeline 3-Level cargado correctamente desde: E:\Futuro\MLAlgoTrading\TradingBots\Models/gc/trading_model_gru_3level
📌 Features: 71
📌 Thresholds cargados: True
📁 Resultados cargados: True


In [5]:
df_results.tail()

,datetime,actual,prob_hold,prob_move,prob_sell,prob_buy,pred_logret,real_logret,pred_move,pred_dir,final_pred
24115,2026-01-13 08:00:00,1.0,0.673578,0.326422,0.675285,0.324715,0.026717,0.027770,0,1,1
24116,2026-01-13 12:00:00,1.0,0.642417,0.357583,0.860814,0.139186,0.022461,0.028560,0,1,1
24117,2026-01-13 16:00:00,2.0,0.625786,0.374214,0.867238,0.132762,0.019205,0.027746,0,1,1
24118,2026-01-13 20:00:00,2.0,0.649779,0.350221,0.866413,0.133587,0.014540,0.018535,0,1,1
24119,2026-01-14 00:00:00,0.0,0.656474,0.343526,0.849640,0.150360,0.010633,0.023068,0,1,1


In [10]:
params

{'input_dim': 71, 'output_dim': 2}

In [12]:
params_g

{'input_dim': 71, 'output_dim': 2}

In [6]:
thresholds

{'threshold_move': 0.7, 'threshold_dir': 0.5}

In [13]:
thresholds_g

{'threshold_move': 0.65, 'threshold_dir': 0.5}

In [7]:
df_final, spec = prepare_all(ASSET, MINUTES, RETURN_HORIZON_MIN, json_config_path="../Data/features_config.json")

In [8]:
df_final

,datetime,open,high,low,close,volume,openint,ticker,per,trading_date,...,fib_dist_786,nearest_fib_level,dist_to_nearest_fib,swing_extension,bars_between_swing_highs,bars_between_swing_lows,swing_high_velocity,swing_low_velocity,bars_between_swings,swing_cycle_ratio
0,2006-12-04 08:00:00,651.2,653.6,651.0,653.4,610,0.0,GC,240min,2006-12-04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2006-12-04 12:00:00,653.4,653.9,651.9,652.3,741,0.0,GC,240min,2006-12-04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2006-12-04 16:00:00,652.2,652.8,650.6,651.6,302,0.0,GC,240min,2006-12-04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2006-12-04 20:00:00,651.5,652.0,649.5,651.0,1106,0.0,GC,240min,2006-12-04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2006-12-05 00:00:00,651.0,651.6,644.5,650.6,9571,0.0,GC,240min,2006-12-04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30566,2026-01-19 12:00:00,4684.3,4685.5,4660.0,4671.2,22930,0.0,GC,240min,2026-01-19,...,0.759029,0.786,0.759029,0.545029,6.0,3.0,14.250000,28.5,0.0,1.666667
30567,2026-01-19 16:00:00,4670.9,4684.3,4666.4,4668.4,18219,0.0,GC,240min,2026-01-19,...,0.027719,0.786,0.027719,-0.186281,3.0,3.0,52.966667,28.5,3.0,0.000000
30568,2026-01-19 20:00:00,4668.2,4678.5,4664.5,4675.7,20254,0.0,GC,240min,2026-01-19,...,0.073660,0.786,0.073660,-0.140340,3.0,3.0,52.966667,28.5,3.0,0.333333
30569,2026-01-20 00:00:00,4676.0,4681.2,4666.3,4677.8,26475,0.0,GC,240min,2026-01-19,...,0.086876,0.786,0.086876,-0.127124,3.0,3.0,52.966667,28.5,3.0,0.666667


In [9]:
df_final[features]

,williams_r,is_ny,adx,return_lag_20,volume_cum,avg_volume,volume_delta,dist_low_cum_ticks,pct_above_ma_20,atr_norm,...,rolling_kurt_10,ad_z,dist_to_last_swing_low,ad_norm,bars_since_swing_low,obv_rel,rsi,roc_5,obv_roc_5,cmo
0,NaN,0,NaN,NaN,610,NaN,NaN,24.0,NaN,NaN,...,NaN,NaN,NaN,1.985207e+02,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,0,NaN,NaN,1351,NaN,131.0,13.0,NaN,0.003066,...,NaN,NaN,NaN,3.577692e+01,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,1,NaN,NaN,1653,NaN,-439.0,10.0,NaN,0.003110,...,NaN,NaN,NaN,2.004514e+01,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,1,NaN,NaN,2759,NaN,804.0,15.0,NaN,0.003210,...,NaN,NaN,NaN,1.061197e+02,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,0,NaN,NaN,12330,NaN,8465.0,61.0,NaN,0.004239,...,NaN,NaN,NaN,1.005668e+03,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30566,-16.865953,0,22.973304,0.004658,62429,36313.05,-16569.0,490.0,0.010995,0.008045,...,8.147275,1.932558,0.028280,8.081741e+05,2.0,-5.451608,65.900208,0.013803,0.033522,27.661851
30567,-18.628068,1,22.745801,0.004507,80648,36102.05,-4711.0,462.0,0.010035,0.007488,...,8.165638,1.074123,0.027697,1.150519e+06,3.0,-5.508702,65.014146,0.016173,0.006312,43.045564
30568,-14.033984,1,22.452681,0.001100,100902,35415.95,2035.0,535.0,0.011235,0.006879,...,7.862779,1.581022,0.029215,1.471889e+06,4.0,-5.586826,66.286847,0.018915,-0.005664,41.321804
30569,-12.712398,0,22.264363,0.000625,127377,35456.85,6221.0,556.0,0.011320,0.006384,...,7.822457,1.998388,0.029651,1.383949e+06,5.0,-5.543047,66.662552,0.016670,-0.011338,38.788660


In [ ]:
import numpy as np

def create_test_dataset(df, features, lookback=30):
    """
    Crea el dataset para LSTM usando NumPy strides (ultra rápido).
    
    Args:
        df: DataFrame con features y targets
        features: list de nombres de columnas de features
        target_col: str o dict. Si es dict, devuelve múltiples targets
        lookback: número de períodos anteriores para la ventana
    
    Returns:
        Si target_col es str: (X, y)
        Si target_col es dict: (X, {target_name: y_array, ...})
    """
    # 1. Convertimos features a NumPy array
    feature_array = df[features].values
    
    # 2. Calculamos las dimensiones
    num_samples = len(df) - lookback
    num_features = len(features)
    
    # 3. Magia de NumPy Strides: Creamos ventanas sin bucles
    shape = (num_samples, lookback, num_features)
    strides = (feature_array.strides[0], feature_array.strides[0], feature_array.strides[1])
    
    X = np.lib.stride_tricks.as_strided(feature_array, shape=shape, strides=strides)
    
    return X

In [20]:
test = create_test_dataset(df_final, features, lookback=30)

In [21]:
test_g = create_test_dataset(df_final, features_g, lookback=30)

In [22]:
test

array([[[            nan,  0.00000000e+00,             nan, ...,
                     nan,             nan,             nan],
        [            nan,  0.00000000e+00,             nan, ...,
                     nan,             nan,             nan],
        [            nan,  1.00000000e+00,             nan, ...,
                     nan,             nan,             nan],
        ...,
        [-5.93750000e+01,  1.00000000e+00,  5.51813842e+01, ...,
          5.05928854e-03, -1.63462971e-01, -3.38403042e+01],
        [-5.72368421e+01,  0.00000000e+00,  5.23281671e+01, ...,
         -1.72738693e-03,  1.22871642e-02, -2.89795918e+01],
        [-9.50617284e+01,  0.00000000e+00,  5.06002009e+01, ...,
         -1.16279070e-02,  2.73153663e-01, -4.18060201e+01]],

       [[            nan,  0.00000000e+00,             nan, ...,
                     nan,             nan,             nan],
        [            nan,  1.00000000e+00,             nan, ...,
                     nan,             

In [23]:
test_flat = test.reshape(-1, test.shape[-1])


In [24]:
test_flat

array([[            nan,  0.00000000e+00,             nan, ...,
                    nan,             nan,             nan],
       [            nan,  0.00000000e+00,             nan, ...,
                    nan,             nan,             nan],
       [            nan,  1.00000000e+00,             nan, ...,
                    nan,             nan,             nan],
       ...,
       [-1.86280680e+01,  1.00000000e+00,  2.27458014e+01, ...,
         1.61729174e-02,  6.31160245e-03,  4.30455635e+01],
       [-1.40339836e+01,  1.00000000e+00,  2.24526807e+01, ...,
         1.89152084e-02, -5.66435784e-03,  4.13218036e+01],
       [-1.27123977e+01,  0.00000000e+00,  2.22643627e+01, ...,
         1.66699268e-02, -1.13381787e-02,  3.87886598e+01]],
      shape=(916230, 71))

In [ ]:
test_scaled =sc_features.transform(test_flat).reshape(test.shape)

In [32]:
import torch
model_l1.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_test_t = torch.tensor(test_scaled, dtype=torch.float32).to(device)
model_l1.eval()
with torch.no_grad():
    logits = model_l1(X_test_t)
    probs = torch.softmax(logits, dim=1)


OutOfMemoryError: CUDA out of memory. Tried to allocate 9.65 GiB. GPU 0 has a total capacity of 6.00 GiB of which 0 bytes is free. Of the allocated memory 1.05 GiB is allocated by PyTorch, and 9.65 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [33]:
def predict_in_batches(model, X_scaled, batch_size=512, device="cuda"):
    model.eval()
    preds = []

    with torch.no_grad():
        for i in range(0, len(X_scaled), batch_size):
            batch = X_scaled[i:i+batch_size]
            batch_t = torch.tensor(batch, dtype=torch.float32).to(device)
            logits = model(batch_t)
            preds.append(logits.cpu())

    return torch.cat(preds, dim=0)


In [35]:
logits_l1 = predict_in_batches(model_l1, test_scaled)
probs_l1 = torch.softmax(logits_l1, dim=1).numpy()


In [36]:
probs_l1

array([[       nan,        nan],
       [       nan,        nan],
       [       nan,        nan],
       ...,
       [0.19215396, 0.80784607],
       [0.4266165 , 0.5733835 ],
       [0.47765824, 0.52234185]], shape=(30541, 2), dtype=float32)